## 1. Mount Google Drive (optional)

Skip this cell if you don't want persistent outputs. If you skip, everything lives under `/content/` and disappears when the runtime ends.

Mounting puts your runs at `/content/drive/MyDrive/swarm_runs/` and your model cache at `/content/drive/MyDrive/swarm_models/` so subsequent sessions start with everything ready.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths

Set the four path-related env vars. Adjust `USE_DRIVE` to choose between Drive (persistent) and `/content/` (ephemeral).

In [ ]:
import os

USE_DRIVE = True  # set False to keep everything under /content/ (ephemeral)

if USE_DRIVE:
    BASE = '/content/drive/MyDrive/swarm'
else:
    BASE = '/content/swarm'

os.environ['SWARM_OUTPUTS_BASE_DIR']     = f'{BASE}/runs'
os.environ['SWARM_MODELS_DIR']           = f'{BASE}/models'
os.environ['SWARM_KB_DIR']               = f'{BASE}/knowledge_base'
os.environ['SWARM_RETRIEVAL_CACHE_DIR']  = f'{BASE}/retrieval_cache'
# HuggingFace cache (model weights + tokenizers). Keep on Drive so reload is free.
os.environ['HF_HOME']                    = f'{BASE}/hf_cache'

for d in ('runs', 'models', 'knowledge_base', 'retrieval_cache', 'hf_cache'):
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

print('SWARM paths:')
for k in sorted(os.environ):
    if k.startswith('SWARM_') or k == 'HF_HOME':
        print(f'  {k} = {os.environ[k]}')

## 3. Clone the repository

We clone into `/content/swarm_repo` (always ephemeral — the repo itself is on GitHub, no need to keep it on Drive). The pipeline cd's into `Attempt At Cleaning/`.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/YOUR_USERNAME/ai_swarm_mechanics.git'  # <-- edit me
REPO_DIR = '/content/swarm_repo'
WORKDIR  = f'{REPO_DIR}/Attempt At Cleaning'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'{REPO_DIR} exists; running git pull')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(WORKDIR)
print('cwd =', os.getcwd())

## 4. Install dependencies

`llama-cpp-python` with CUDA-12.1 prebuilt wheels for fast GGUF inference. Plus the optional retrieval and embedding stacks.

In [ ]:
!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 llama-cpp-python
!pip install -q sentence-transformers wikipedia beautifulsoup4 requests tqdm huggingface_hub

## 5. Download GGUF models

Edit `MODELS` if you want different model files or different Hugging Face source repos. The default values use bartowski's GGUF mirrors, which publish the Q4_K_M / Q5_K_M variants the pipeline expects.

The filenames here must match `configs/heterogeneous.json`. Total ~33 GB — if you're on Colab free tier and `USE_DRIVE=False`, skip the 14B synth model and edit the manifest below to use a smaller checkpoint for the synthesizer slot.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

MODELS_DIR = Path(os.environ['SWARM_MODELS_DIR'])
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# {target_filename: (hf_repo_id, filename_within_repo)}
# Adjust if any of these repos move; bartowski's mirrors are reasonably stable.
MODELS = {
    'Qwen2.5-7B-Instruct-Q5_K_M.gguf':           ('bartowski/Qwen2.5-7B-Instruct-GGUF',           'Qwen2.5-7B-Instruct-Q5_K_M.gguf'),
    'Mistral-Nemo-Instruct-2407-Q4_K_M.gguf':    ('bartowski/Mistral-Nemo-Instruct-2407-GGUF',    'Mistral-Nemo-Instruct-2407-Q4_K_M.gguf'),
    'Phi-3.5-mini-instruct-Q4_K_M.gguf':         ('bartowski/Phi-3.5-mini-instruct-GGUF',         'Phi-3.5-mini-instruct-Q4_K_M.gguf'),
    'Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf':    ('bartowski/Meta-Llama-3.1-8B-Instruct-GGUF',    'Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf'),
    'DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf':   ('bartowski/DeepSeek-R1-Distill-Qwen-7B-GGUF',   'DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf'),
    'Qwen2.5-14B-Instruct-Q4_K_M.gguf':          ('bartowski/Qwen2.5-14B-Instruct-GGUF',          'Qwen2.5-14B-Instruct-Q4_K_M.gguf'),
}

for target, (repo, src) in MODELS.items():
    dst = MODELS_DIR / target
    if dst.exists():
        print(f'[skip] {target} already in {MODELS_DIR}')
        continue
    print(f'[get ] {repo}/{src} -> {dst.name}')
    path = hf_hub_download(repo_id=repo, filename=src, local_dir=str(MODELS_DIR))
    # hf_hub_download may write to a snapshot subdir; symlink/rename if needed.
    if Path(path).resolve() != dst.resolve():
        try:
            Path(path).rename(dst)
        except OSError:
            os.symlink(path, dst)

print('\nmodels directory:')
for f in sorted(MODELS_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name:50s}  {f.stat().st_size / 1e9:6.2f} GB')

## 6. Quick smoke test (MockLLM, no model load)

Verifies the pipeline + env vars before you commit to a full real-model run.

In [ ]:
!MOCK_LLM=1 python run_swarm.py debate "Test thesis" --heterogeneous --corpus=placeholder

## 7. Real run — in-process heterogeneous routing

Loads each role's model in turn within one Python process. Works well on Colab T4 (16 GB VRAM) — the in-process unload that's flaky on Windows-with-16-GB-RAM is reliable here. Expect ~90–120 min for a 3-round debate.

In [ ]:
!python run_swarm.py debate "Does free will exist?" --heterogeneous

## 8. Alternative — phase-isolated orchestrator

Spawns 19 short-lived subprocesses (one per phase per round + synth). Each loads exactly one model, exits, and the next picks up the SignalStore checkpoint. Slower (subprocess startup overhead) but **crash-resumable** — if a phase fails, re-run the same command and it skips completed phases.

Use this if you hit Colab GPU OOM or want resumability.

In [ ]:
!python tools/run_isolated.py debate "Does free will exist?" --heterogeneous --run-id=free_will_isolated

## 9. Inspect outputs

Outputs are at `$SWARM_OUTPUTS_BASE_DIR/outputs/<run_id>/`. Each run dir contains `answer.txt`, `signals.json`, `summary.json`, `round_log.json`, `citations.json`, `lineage.dot`, `run_meta.json`.

In [ ]:
import glob, json
from pathlib import Path

runs = sorted(Path(os.environ['SWARM_OUTPUTS_BASE_DIR'], 'outputs').glob('*'),
              key=lambda p: p.stat().st_mtime)
if runs:
    latest = runs[-1]
    print('latest run:', latest)
    print()
    print('=== summary.json ===')
    print(json.dumps(json.loads((latest / 'summary.json').read_text()), indent=2))
    print()
    print('=== answer.txt ===')
    print((latest / 'answer.txt').read_text())
else:
    print('no runs yet')

## Troubleshooting

- **Colab session disconnects mid-run.** If `USE_DRIVE=True`, your store_state.json checkpoints are safe under `$SWARM_OUTPUTS_BASE_DIR/outputs/<run_id>/`. Reconnect and re-run the orchestrator with the same `--run-id` — it skips completed phases.
- **`llama-cpp-python` install fails.** The CUDA wheel URL may have moved. Try the CPU wheel: `!pip install llama-cpp-python` (much slower).
- **HuggingFace download stalls.** Some bartowski mirrors are rate-limited; retry, or substitute another GGUF re-uploader's repo in the `MODELS` dict.
- **VRAM OOM on the 14B synth.** Edit `configs/heterogeneous.json` and change `"synthesizer"` to a 7B-class file you already downloaded.
- **Drive quota.** The full six-model set is ~33 GB. If your Drive is tight, skip a few models — the pipeline falls back to MockLLM for missing slots (visible in the `[router] WARNING` line), but those agents will produce nothing useful.